# Sesión 3 · Transformers por dentro

Abre el pipeline y controla cada paso a mano.

Objetivos:
1. Reproducir a mano una clasificación (tokenizar, inferir, softmax).
2. Ver las tres familias.
3. Construir un buscador semántico.

In [ ]:
%pip install -q transformers sentence-transformers

## 1 · Reproducir a mano un pipeline de clasificación

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

nombre = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(nombre)
modelo = AutoModelForSequenceClassification.from_pretrained(nombre)

texto = "I really enjoyed this movie!"

# 1) tokenizar
entradas = tokenizer(texto, return_tensors="pt")
# 2) inferir
with torch.no_grad():
    logits = modelo(**entradas).logits
# 3) softmax + argmax
probs = torch.softmax(logits, dim=-1)
pred = int(torch.argmax(probs, dim=-1))
print("logits:", logits)
print("probabilidades:", probs)
print("etiqueta:", modelo.config.id2label[pred])

## 2 · Las tres familias (un vistazo a la arquitectura)

- **Encoder** (BERT): lee todo, ideal para clasificar.
- **Decoder** (GPT): genera token a token.
- **Encoder-decoder** (T5): transforma (traducción, resumen).

In [ ]:
from transformers import AutoModel, AutoModelForCausalLM, AutoModelForSeq2SeqLM

# Tres familias, tres clases distintas
_enc  = AutoModel.from_pretrained("distilbert-base-uncased")
_dec  = AutoModelForCausalLM.from_pretrained("gpt2")
_encdec = AutoModelForSeq2SeqLM.from_pretrained("t5-small")
print("encoder (BERT), decoder (GPT-2) y encoder-decoder (T5) cargados")

## 3 · Embeddings de frase

In [ ]:
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer("all-MiniLM-L6-v2")
emb = modelo.encode(["¿Dónde está mi pedido?", "Estado de mi envío", "El cielo es azul"])
print("dimensión de cada embedding:", emb.shape[1])

## Reto: buscador semántico sobre FAQs

Codifica 100 preguntas frecuentes y, ante una consulta, devuelve la más parecida por similitud coseno.

In [ ]:
import numpy as np

faqs = [
    "¿Cuánto tarda el envío?",
    "¿Cómo puedo devolver un producto?",
    "¿Aceptáis pagos con tarjeta?",
    "¿Cuál es el horario de atención?",
]
consulta = "¿Me podéis decir el plazo de entrega?"

# TODO: codifica faqs y consulta, calcula similitud coseno y devuelve la mejor FAQ
# pista: coseno = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))